In [1]:
import scanpy as sc
import tempfile
import requests

def load_from_url(url):
    response = requests.get(url)
    with tempfile.NamedTemporaryFile(suffix=".h5ad", delete=False) as tmp_file:
        tmp_file.write(response.content)
        tmp_file.flush()
        return sc.read_h5ad(tmp_file.name)

# Load AD and cerebellum data
ad_url = "https://datasets.cellxgene.cziscience.com/42f1ec32-aaa2-4f08-8f78-a34f9b993ea8.h5ad"
cereb_url = "https://datasets.cellxgene.cziscience.com/b07a1ded-bf24-48ef-b32b-a58de97b8080.h5ad"

adata = load_from_url(ad_url)
cereb_data = load_from_url(cereb_url)

In [2]:
shared_genes = list(set(adata.var_names) & set(cereb_data.var_names))
adata_sub = adata[:, shared_genes]
cereb_sub = cereb_data[:, shared_genes]

cereb_sub

View of AnnData object with n_obs × n_vars = 180956 × 27409
    obs: 'orig_cluster', 'orig_sub_cluster', 'broad_lineage', 'author_cell_type', 'dev_state', 'subtype', 'precisest_label', 'tissue_id', 'batch', 'size_factor', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'is_primary_data', 'author_stage', 'tissue_fragment', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'batch_condition', 'citation', 'default_embedding', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_liger', 'X_umap2d', 'X_umap3d'

In [3]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

X_ad = adata_sub.X
y_ad = LabelEncoder().fit_transform(adata_sub.obs["disease"])  # e.g., AD vs Control

X_cereb = cereb_sub.X
if "development_stage" in cereb_sub.obs.columns:
    y_cereb = LabelEncoder().fit_transform(cereb_sub.obs["development_stage"])
else:
    y_cereb = np.zeros(X_cereb.shape[0])

print(X_ad)

#print(y_ad)

  (0, 1238)	1.8974599838256836
  (0, 16642)	2.5126731395721436
  (0, 23270)	1.8974599838256836
  (0, 6384)	2.5126731395721436
  (0, 434)	1.8974599838256836
  (0, 5511)	1.8974599838256836
  (0, 1648)	1.8974599838256836
  (0, 4191)	2.5126731395721436
  (0, 2659)	1.8974599838256836
  (0, 8257)	1.8974599838256836
  (0, 19008)	1.8974599838256836
  (0, 5061)	1.8974599838256836
  (0, 1667)	1.8974599838256836
  (0, 19479)	1.8974599838256836
  (0, 13088)	2.5126731395721436
  (0, 10749)	1.8974599838256836
  (0, 3747)	1.8974599838256836
  (0, 27384)	1.8974599838256836
  (0, 25647)	1.8974599838256836
  (0, 23210)	1.8974599838256836
  (0, 3431)	1.8974599838256836
  (0, 21784)	2.890749454498291
  (0, 1928)	2.5126731395721436
  (0, 6063)	1.8974599838256836
  (0, 26933)	2.5126731395721436
  :	:
  (24341, 9726)	0.7869114279747009
  (24341, 23990)	1.2217743396759033
  (24341, 4296)	1.0277972221374512
  (24341, 18000)	0.4689410924911499
  (24341, 17209)	1.2217743396759033
  (24341, 25666)	1.6463686227798

In [4]:
from sklearn.svm import LinearSVC
from sklearn.feature_selection import SelectFromModel

# SVM for AD
svm_ad = LinearSVC(C=0.01, penalty='l1', dual=False, max_iter=5000)
svm_ad.fit(X_ad, y_ad)
selected_ad = np.abs(svm_ad.coef_).sum(axis=0)

# SVM for Cerebellum
svm_cereb = LinearSVC(C=0.01, penalty='l1', dual=False, max_iter=5000)
svm_cereb.fit(X_cereb, y_cereb)
selected_cereb = np.abs(svm_cereb.coef_).sum(axis=0)

# Top genes
top_n = 50
genes_array = np.array(shared_genes)
top_ad_genes = genes_array[np.argsort(selected_ad)[-top_n:]]
top_cereb_genes = genes_array[np.argsort(selected_cereb)[-top_n:]]
overlap = set(top_ad_genes) & set(top_cereb_genes)
print(f"Overlap ({len(overlap)}): {overlap}")


Overlap (7): {'ENSG00000280441', 'ENSG00000228696', 'ENSG00000230876', 'ENSG00000210082', 'ENSG00000183878', 'ENSG00000109846', 'ENSG00000204389'}


/users/imbahndu/.local/lib/python3.9/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [5]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report

# Combine inputs
X_combined = np.vstack([X_ad, X_cereb])
y_ad_combined = np.concatenate([y_ad, [-1]*X_cereb.shape[0]])
y_cereb_combined = np.concatenate([[-1]*X_ad.shape[0], y_cereb])

# Train separate MLPs (simplified multitask)
mlp_ad = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200)
mlp_ad.fit(X_ad, y_ad)
print("AD classification report:")
print(classification_report(y_ad, mlp_ad.predict(X_ad)))

mlp_cereb = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=200)
mlp_cereb.fit(X_cereb, y_cereb)
print("Cerebellum classification report:")
print(classification_report(y_cereb, mlp_cereb.predict(X_cereb)))


AD classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      6020
           1       1.00      1.00      1.00      8939
           2       1.00      1.00      1.00      9383

    accuracy                           1.00     24342
   macro avg       1.00      1.00      1.00     24342
weighted avg       1.00      1.00      1.00     24342

Cerebellum classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13445
           1       1.00      1.00      1.00     13197
           2       1.00      1.00      1.00      7849
           3       1.00      1.00      1.00     14200
           4       1.00      1.00      1.00     15580
           5       1.00      1.00      1.00      4594
           6       1.00      1.00      1.00      8033
           7       1.00      1.00      1.00      9359
           8       1.00      1.00      1.00      3181
           9      

In [ ]:
from sklearn.inspection import permutation_importance

perm_ad = permutation_importance(mlp_ad, X_ad.toarray(), y_ad, n_repeats=10, random_state=42)
top_nn_ad = genes_array[np.argsort(perm_ad.importances_mean)[-top_n:]]

perm_cereb = permutation_importance(mlp_cereb, X_cereb.toarray(), y_cereb, n_repeats=10, random_state=42)
top_nn_cereb = genes_array[np.argsort(perm_cereb.importances_mean)[-top_n:]]

overlap_nn = set(top_nn_ad) & set(top_nn_cereb)
print(f"Overlap in NN-important genes: {overlap_nn}")